In [19]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv("../data/processed/wdi_cleaned.csv")
# Sicherstellen, dass Year ein Integer ist für die Zeitachsen
df['Year'] = df['Year'].astype(int)

In [20]:
# Prüfen, wie viele Länder pro Jahr Daten für Internet haben
check_coverage = df.dropna(subset=['internet_users']).groupby('Year').size()
print("Datenabdeckung pro Jahr:")
print(check_coverage.tail(10)) # Zeigt die letzten 10 Jahre

Datenabdeckung pro Jahr:
Year
2005    237
2006    237
2007    245
2008    244
2009    244
2010    245
2011    247
2012    244
2013    243
2014    242
dtype: int64


In [21]:
# 1. Daten für das Jahr 2014 filtern
df_2014 = df[df['Year'] == 2014].copy()

# 2. Karte zeichnen
fig1 = px.choropleth(
    df_2014, 
    locations="Country Code",
    locationmode="ISO-3",
    color="internet_users", 
    hover_name="Country Name",
    title="Globale Internetnutzung im Jahr 2014 (%)",
    color_continuous_scale=px.colors.sequential.Viridis,
    labels={'internet_users': 'Internetnutzer (%)'}
)

fig1.update_layout(
    geo=dict(projection_type='natural earth'),
    margin={"r":0,"t":50,"l":0,"b":0}
)

fig1.show()

In [22]:
print(df[["gdp_per_capita"]])

      gdp_per_capita
0                NaN
1         184.494712
2         195.776630
3         216.708129
4         247.664140
...              ...
3603      854.847665
3604      980.208824
3605     1163.418689
3606     1268.126633
3607     1264.983826

[3608 rows x 1 columns]


In [23]:
# 1. Kopie der 2014er Daten erstellen
df_scatter = df_2014.copy()

# 2. Bereinigung: Nur Zeilen behalten, die in ALLEN drei Kategorien Daten haben
# Plotly braucht gültige Zahlen für x, y und size
df_scatter = df_scatter.dropna(subset=['internet_users', 'co2_per_capita', 'gdp_per_capita'])

# 3. Negative Werte ausschließen (besonders für die 'size' Spalte wichtig)
df_scatter = df_scatter[
    (df_scatter['gdp_per_capita'] > 0) & 
    (df_scatter['co2_per_capita'] >= 0) & 
    (df_scatter['internet_users'] >= 0)
]

print(f"Datenpunkte für Scatterplot nach Bereinigung: {len(df_scatter)}")

# 4. Der Scatter Plot
fig3 = px.scatter(
    df_scatter, 
    x="internet_users", 
    y="co2_per_capita",
    size="gdp_per_capita",
    color="Country Name",
    hover_name="Country Name",
    trendline="ols",
    title="Zusammenhang: Internetnutzung vs. CO2-Emissionen (2014)",
    labels={
        "internet_users": "Internetnutzer (%)",
        "co2_per_capita": "CO2-Emissionen pro Kopf (Tonnen)",
        "gdp_per_capita": "BIP pro Kopf"
    },
    size_max=60 # Damit die Blasen nicht den ganzen Bildschirm füllen
)

fig3.show()

Datenpunkte für Scatterplot nach Bereinigung: 237
